# 00 · Exploración guiada (Hito 1)
Reportes diarios de JHU CSSE, enero a junio de 2020, leídos directamente desde el repositorio (sin descargar archivos).

| Tarea del enunciado | Dónde |
|---|---|
| 1 · Cargar `01-22-2020.csv`: primeras filas, filas y columnas, tipos | Sección 1 |
| 2 · Nulos por columna y qué significan | Sección 3 |
| 3 · snake_case y `last_update` en AAAA-MM-DD | Sección 2 |
| 4 · Unir enero a junio (dos cambios de esquema) | Sección 2 |
| 5 · Top 10 al 30-jun y evolución de Chile | Secciones 4 y 5 |
| 6 · Perfilado automático y tres observaciones | Sección 6 |

## 1 · Primer reporte: `01-22-2020.csv`
Antes de unir todo, miramos un solo archivo para conocer el formato original (esquema 1).

In [ ]:
import pandas as pd

BASE = (
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
    "csse_covid_19_data/csse_covid_19_daily_reports/"
)

primer_reporte = pd.read_csv(BASE + "01-22-2020.csv")

display(primer_reporte.head())
print(f"Filas: {primer_reporte.shape[0]} | Columnas: {primer_reporte.shape[1]}")
print("\nTipos de datos:")
print(primer_reporte.dtypes)
print("\nPaíses en el primer reporte:", primer_reporte["Country/Region"].nunique())


**Qué vemos:** 76 filas y 6 columnas, con 22 países o entidades (31 filas son provincias de China), y nombres del esquema 1 (`Province/State`, `Country/Region`, `Last Update`).
`Last Update` llega como texto (`object` en pandas 2, `str` en pandas 3), no como fecha. `Confirmed`, `Deaths` y `Recovered` son `float64`
aunque son conteos: pandas usa `float` porque hay celdas vacías (NaN) y un `int` normal no admite NaN.

## 2 · Unir enero a junio de 2020 y estandarizar columnas
Leemos los 161 reportes (22-ene → 30-jun). En este período aparecen dos cambios de esquema:
el 1-mar llegan `Latitude`/`Longitude` (esquema 2) y el 22-mar cambia todo el formato (esquema 3: `FIPS`, `Admin2`, `Lat`, `Long_`, `Combined_Key`…).
El diccionario `RENOMBRAR` lleva todos los nombres a una sola versión en snake_case.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

# Leemos directo desde el repositorio de JHU (raw), sin descargar archivos.
BASE = (
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
    "csse_covid_19_data/csse_covid_19_daily_reports/"
)

# Un solo diccionario traduce los nombres de los 5 esquemas a snake_case.
# E2: "Long_" (esquema 3+) y las dos variantes de la tasa de incidencia van a UNA columna.
RENOMBRAR = {
    "Province/State": "province_state", "Province_State": "province_state",
    "Country/Region": "country_region", "Country_Region": "country_region",
    "Last Update": "last_update", "Last_Update": "last_update",
    "Latitude": "lat", "Lat": "lat", "Longitude": "long", "Long_": "long",
    "Incidence_Rate": "incident_rate", "Incident_Rate": "incident_rate",
    "Case-Fatality_Ratio": "case_fatality_ratio", "Case_Fatality_Ratio": "case_fatality_ratio",
}


def leer_reporte(fecha):
    """Lee el reporte de una fecha desde raw y normaliza sus columnas."""
    nombre = fecha.strftime("%m-%d-%Y") + ".csv"   # el archivo se llama mes-día-año
    t = pd.read_csv(BASE + nombre).rename(columns=RENOMBRAR)
    t.columns = t.columns.str.lower()
    t["fecha_reporte"] = fecha                      # fecha del dato = la del archivo (no last_update)
    return t


# E1: los reportes parten el 22-ene-2020 (antes no hay archivos -> 404).
fechas = pd.date_range("2020-01-22", "2020-06-30", freq="D")

# 10 lecturas en paralelo; ex.map respeta el orden de las fechas.
with ThreadPoolExecutor(max_workers=10) as ex:
    df = pd.concat(list(ex.map(leer_reporte, fechas)), ignore_index=True)

print(f"Reportes leídos: {len(fechas)} | filas: {df.shape[0]} | columnas: {df.shape[1]}")
display(df.head())


In [ ]:
# Alternativa sin hilos (como la PPT S14), ~1 min para 161 archivos:
# df = pd.concat([leer_reporte(f) for f in fechas], ignore_index=True)

### Normalización de nombres de país (E3)
JHU cambió cómo escribía algunos países (`Mainland China` → `China`, `South Korea` → `Korea, South`).
Además, algunos territorios partieron como si fueran países y luego pasaron a ser provincias
de otro (por ejemplo `Hong Kong` → provincia de `China`). Si no los unificamos, un mismo país
queda partido en dos y sus casos se cuentan mal.

In [ ]:
PAIS = {
    " Azerbaijan": "Azerbaijan", "Bahamas, The": "Bahamas", "The Bahamas": "Bahamas",
    "Cape Verde": "Cabo Verde", "Czech Republic": "Czechia", "East Timor": "Timor-Leste",
    "Gambia, The": "Gambia", "The Gambia": "Gambia", "Iran (Islamic Republic of)": "Iran",
    "Ivory Coast": "Cote d'Ivoire", "Mainland China": "China", "Republic of Ireland": "Ireland",
    "Republic of Korea": "Korea, South", "South Korea": "Korea, South",
    "Republic of Moldova": "Moldova", "Republic of the Congo": "Congo (Brazzaville)",
    "Russian Federation": "Russia", "Taiwan": "Taiwan*", "Taipei and environs": "Taiwan*",
    "Vatican City": "Holy See", "Viet Nam": "Vietnam",
    "Palestine": "West Bank and Gaza", "occupied Palestinian territory": "West Bank and Gaza",
    "North Ireland": "United Kingdom", "Others": "Diamond Princess", "Cruise Ship": "Diamond Princess",
}

# Territorios que primero venían como "país" y después como provincia de otro país.
TERRITORIO = {
    "Hong Kong": "China", "Hong Kong SAR": "China", "Macau": "China", "Macao SAR": "China",
    "Puerto Rico": "US", "Guam": "US",
    "Aruba": "Netherlands", "Curacao": "Netherlands",
    "Cayman Islands": "United Kingdom", "Channel Islands": "United Kingdom",
    "Gibraltar": "United Kingdom", "Jersey": "United Kingdom", "Guernsey": "United Kingdom",
    "Faroe Islands": "Denmark", "Greenland": "Denmark",
    "French Guiana": "France", "Guadeloupe": "France", "Martinique": "France", "Mayotte": "France",
    "Reunion": "France", "Saint Barthelemy": "France", "Saint Martin": "France", "St. Martin": "France",
}

es_territorio = df["country_region"].isin(TERRITORIO)
df.loc[es_territorio, "province_state"] = df.loc[es_territorio, "country_region"]
df.loc[es_territorio, "country_region"] = df.loc[es_territorio, "country_region"].map(TERRITORIO)
df["country_region"] = df["country_region"].replace(PAIS)

# Control: ningún nombre debería quedar "huérfano" (existir antes pero no al 30-jun).
paises_fin = set(df.loc[df["fecha_reporte"] == "2020-06-30", "country_region"])
print("Nombres sin equivalente al 30-jun:", sorted(set(df["country_region"]) - paises_fin))


### last_update a formato AAAA-MM-DD
`last_update` viene con dos formatos mezclados (`1/22/2020 17:00` y `2020-07-01 04:33:52`).
Ojo: es **cuándo JHU tocó la fila**, no la fecha del dato (esa es `fecha_reporte`).

In [ ]:
df["last_update"] = pd.to_datetime(df["last_update"], format="mixed").dt.normalize()

# Observación de calidad (oportunidad): hay filas de reportes de 2020 actualizadas mucho después.
print("last_update mínimo:", df["last_update"].min().date(), "| máximo:", df["last_update"].max().date())
df[["fecha_reporte", "last_update"]].head()


In [ ]:
import numpy as np

# Esquema según la fecha del reporte. De enero a junio de 2020 solo existen los esquemas 1, 2 y 3.
# np.select toma la primera condición que se cumple.
condiciones = [
    df["fecha_reporte"] < "2020-03-01",   # esquema 1: formato original
    df["fecha_reporte"] < "2020-03-22",   # esquema 2: llegan Latitude/Longitude
]
df["esquema"] = np.select(condiciones, [1, 2], default=3)   # desde el 22-mar: esquema 3

df["esquema"].value_counts().sort_index()

In [ ]:
df.shape


In [ ]:
df.describe()


In [ ]:
df.info()

## 3 · Valores nulos por columna

In [ ]:
print(df.isna().sum())

In [ ]:
# Nulos por esquema: muestra si un nulo es "falta el dato" o "la columna aún no existía".
df.isna().groupby(df["esquema"]).sum().T


### Qué significan los nulos en este conjunto
La mayoría **no son datos perdidos**: son columnas que todavía no existían o que no aplican a esa fila.
Por eso no se imputan.

| Columna | Nulos | Significado |
|---|---|---|
| `province_state` | ~22 mil | El país informa como un todo, sin subdivisión (ej. Chile hasta el 19-may). No es un error. |
| `confirmed`, `deaths`, `recovered` | 28 / 432 / 385 | Solo en el esquema 1 (enero–febrero): JHU dejaba la celda vacía cuando no había casos informados (ej. Gansu el 22-ene). |
| `lat`, `long` | ~10,7 mil | Todo el esquema 1 (la columna no existía) más territorios sin coordenadas (ej. filas «Unknown» o «Recovered»). |
| `fips`, `admin2` | ~55 mil | Códigos de **condado de EE.UU.**: no existen antes del 22-mar y no aplican a ningún otro país. |
| `active`, `combined_key` | ~9,8 mil | Columnas creadas en el esquema 3 (22-mar): antes no existían. |
| `incident_rate`, `case_fatality_ratio` | ~230 mil | Columnas del esquema 4 (desde el 29-may): no existen en el 65 % de las filas del período. |

**Conclusión:** solo los nulos de conteo del esquema 1 son "dato faltante" real, y son pocos (menos de 0,2 % de las filas).

## 4 · De acumulados a casos diarios
Los reportes traen **acumulados**: los casos nuevos se calculan restando el día anterior.

**Por qué no restar fila contra fila (E4):** EE.UU. tiene miles de filas por día y Chile 17 desde el 20-may-2020.
Al ordenar por país y fecha, `diff()` restaba una provincia contra otra.

**Qué hacemos:** primero sumamos todas las filas del país por día y después
aplicamos `diff()`. Así el paso de Chile nacional → 16 regiones no produce saltos. El primer día se rellena con
`confirmed`, porque antes del primer reporte el país tenía 0 casos.

El cálculo por territorio (provincia/condado) y la marca de acumulados que retroceden quedan para R2 (H2.1).

In [ ]:
# Nivel PAÍS: sumar primero, diff después
pais = (
    df.groupby(["country_region", "fecha_reporte"], as_index=False)[["confirmed", "deaths"]]
      .sum(min_count=1)                        # min_count=1: si todo es NaN, queda NaN (no 0)
      .sort_values(["country_region", "fecha_reporte"])
)
g = pais.groupby("country_region")
pais["casos_nuevos"] = g["confirmed"].diff()
pais["fallecidos_nuevos"] = g["deaths"].diff()

primer = g.cumcount() == 0                     # primera fila de cada país
pais.loc[primer, "casos_nuevos"] = pais.loc[primer, "confirmed"]
pais.loc[primer, "fallecidos_nuevos"] = pais.loc[primer, "deaths"]

# Control: la suma de los diarios de Chile debe dar su acumulado al 30-jun
chile = pais[pais["country_region"] == "Chile"].set_index("fecha_reporte")
print("Chile · suma de casos nuevos:", int(chile["casos_nuevos"].sum()),
      "| acumulado al 30-jun:", int(chile["confirmed"].iloc[-1]))

### Top 10 países al 30 de junio de 2020 (E5)
Para "casos acumulados al 30-jun" basta sumar `confirmed` de ese día por país.
Sumar `casos_nuevos` no sirve como atajo: la suma de diferencias da *último − primero*,
y con varias provincias por país eso era el acumulado de una sola provincia cualquiera.

In [ ]:
top10 = (
    pais[pais["fecha_reporte"] == "2020-06-30"]
    .nlargest(10, "confirmed")[["country_region", "confirmed"]]
    .reset_index(drop=True)
)
top10


## 5 · Evolución de Chile (22-ene → 30-jun-2020)
Graficamos los **casos nuevos diarios** con su **media móvil de 7 días** (la media se calcula sobre los diarios, nunca sobre los acumulados).
Abajo, el acumulado como referencia.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

chile = pais[pais["country_region"] == "Chile"].set_index("fecha_reporte")
chile["media_movil_7d"] = chile["casos_nuevos"].rolling(7).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})

ax1.bar(chile.index, chile["casos_nuevos"], color="#b9c6d6", label="Casos nuevos diarios")
ax1.plot(chile.index, chile["media_movil_7d"], color="#c62828", lw=2.2, label="Media móvil 7 días")
ax1.axvline(pd.Timestamp("2020-05-20"), color="gray", ls="--", lw=1)
ax1.text(pd.Timestamp("2020-05-18"), ax1.get_ylim()[1] * 0.45,
         "20-may: Chile pasa a informar\npor regiones", fontsize=9, color="gray", ha="right")
ax1.annotate("17-jun: carga de casos\natrasados (rezago)",
             xy=(pd.Timestamp("2020-06-17"), chile.loc["2020-06-17", "casos_nuevos"]),
             xytext=(pd.Timestamp("2020-05-23"), chile["casos_nuevos"].max() * 0.8),
             arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)
ax1.set_title("Chile · casos nuevos de COVID-19, enero a junio de 2020")
ax1.set_ylabel("Casos por día")
ax1.legend(loc="upper left")

ax2.plot(chile.index, chile["confirmed"], color="#37474f", lw=2)
ax2.set_ylabel("Acumulado")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f} mil"))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Primer caso:", chile["confirmed"].gt(0).idxmax().date())
print("Acumulado al 30-jun:", int(chile["confirmed"].iloc[-1]))
print("Peor día:", chile["casos_nuevos"].idxmax().date(), "con", int(chile["casos_nuevos"].max()), "casos")
print("Media móvil 7 días más alta:", chile["media_movil_7d"].idxmax().date(),
      "con", round(chile["media_movil_7d"].max()), "casos/día")


**Lectura del gráfico:** la primera ola de Chile sube con fuerza desde mayo y se mantiene en torno a 5.000–6.000 casos diarios en junio.
El pico aislado del 17-jun (36.179) **no es un día real de contagios**: ese día se sumaron casos que no se habían informado antes.
Fíjense en la media móvil: el rezago la infla durante exactamente 7 días (la "meseta" de ~10.000 entre el 17 y el 23-jun) y luego cae de golpe.
Esa meseta es un artefacto del dato, no una ola: una alerta temprana ingenua se dispararía aquí. Es un problema de calidad que se trata en R2.
El cambio a regiones del 20-may no produce saltos porque sumamos por país antes de calcular los diarios (opción C).

## 6 · Perfilado automático (ydata-profiling)
`minimal=True` omite los cálculos más pesados (correlaciones e interacciones), necesario con 350 mil filas.
El informe se guarda como `perfilado_ene_jun_2020.html`.

In [ ]:
# En Colab hay que instalarlo una vez por sesión.
%pip install -q ydata-profiling


In [ ]:
from ydata_profiling import ProfileReport

perfil = ProfileReport(df, title="Reportes JHU · enero a junio 2020", minimal=True)
perfil.to_file("perfilado_ene_jun_2020.html")
perfil.to_notebook_iframe()


In [ ]:
# Verificamos con código las cifras de las tres observaciones.
fallecidos_mayor = df[df["deaths"] > df["confirmed"]]
print("1) Filas con fallecidos > casos:", len(fallecidos_mayor),
      f"| con casos = 0: {(fallecidos_mayor['confirmed'] == 0).mean():.0%}")
print(fallecidos_mayor["country_region"].value_counts().head(3).to_string())
print("   Filas US 'Unassigned':", (fallecidos_mayor["admin2"] == "Unassigned").sum())

desde_esq3 = df["esquema"] >= 3
print(f"\n2) recovered = 0 desde el esquema 3: {(df.loc[desde_esq3, 'recovered'] == 0).mean():.0%} de las filas")

print(f"\n3) confirmed · mediana: {df['confirmed'].median():.0f} | máximo: {df['confirmed'].max():,.0f}"
      f" | asimetría: {df['confirmed'].skew():.1f}")

atraso = (df["last_update"] - df["fecha_reporte"]).dt.days
print(f"\nExtra) Filas con last_update más de 2 días después del reporte: {(atraso > 2).sum()}"
      f" | máximo: {df['last_update'].max().date()}")


### Tres observaciones del perfilado
1. **Fallecidos sin casos (consistencia).** Hay ~1.360 filas donde `deaths > confirmed`; en el 85 % los casos son 0.
   Casi todas son de EE.UU. (filas `Unassigned`: fallecidos que aún no se asignaban a un condado) y del Reino Unido.
   No es un error de cálculo sino una forma de informar: al sumar por país se compensa. Lo registraremos como regla de consistencia en R2.
2. **`recovered` casi no se informa (completitud).** Desde el esquema 3, alrededor del 90 % de las filas tiene `recovered = 0`
   porque los condados de EE.UU. no informan recuperados. Un 0 aquí significa "no informado", no "nadie se recuperó":
   no usaremos esta columna para calcular casos activos.
3. **Distribución muy asimétrica.** `confirmed` tiene mediana 30 pero máximo 405.843 (asimetría ≈ 16): conviven condados pequeños
   con países enteros. Los promedios engañan; para comparar territorios usaremos tasas por 100.000 habitantes.

**Extra (oportunidad):** 112 filas tienen `last_update` posterior a su fecha de reporte; la más tardía es de 2021-04-02 (Alaska, reporte del 6-may-2020).
JHU corrigió datos viejos casi un año después: por eso la fecha del dato es `fecha_reporte`, nunca `last_update`.

## 7 · Pregunta propia (P7) y fuente externa
**P7:** ¿Los países que llegaron antes al 50 % de población con esquema completo tuvieron menos fallecidos
por 100.000 durante la ola de ómicron (1-dic-2021 a 31-mar-2022)?

- **Fuente externa:** Our World in Data, vacunación, leída desde
  `raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv`.
- **Cruce:** `iso_code` de OWID con el código ISO3 de la tabla UID de JHU.
- **50 %** = `people_fully_vaccinated_per_hundred` ≥ 50 (esquema completo).
- **Ventana de ómicron:** fija e igual para todos los países, 2021-12-01 → 2022-03-31.
- **Cautela:** correlación no es causalidad; los países que se vacunaron antes también suelen tener más ingreso, más testeo
  y otra estructura de edad, y todo eso cambia sus cifras de fallecidos.